<a href="https://colab.research.google.com/github/saphirarivera03-cpu/ClimateChange_SaphiraRivera/blob/main/Saphira_Rivera_ClimateChange.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Climate Change Team — Combined Data Investigation Notebook
## Part A: Week 2 (Define) — Data Cleaning & Visualization
## Part B: Week 4 (Prototype) — Machine Learning & Model Comparison

**Dataset:** Our World in Data — CO2 and Greenhouse Gas Emissions

This notebook contains two work sessions from your Capstone Data Investigation, combined into a single file for easy reference:

- **Part A (Week 2, Wednesday):** Assess → Clean → Visualize → Tell the Story → HCD Design Sprint #2 (Problem Statement)
- **Part B (Week 4):** Define target → Feature engineering → Baseline → Multi-algorithm comparison → Fairness check → HCD Design Sprint #4 (Prototype)

**Run Part A first, top to bottom** — it produces the cleaned dataset and saved CSV that Part B depends on. Part B also includes a fallback that re-cleans from scratch if you skip straight to it, but running Part A first is strongly recommended so your cleaning decisions carry through consistently.

---


---
# PART A — Week 2, Day 3 (Wednesday): Data Cleaning & Visualization + HCD Design Sprint #2
---

# Week 2, Day 3 (Wednesday) — Climate Change Team
## Colab Lab: Data Cleaning & Visualization + HCD Design Sprint #2

**Dataset:** Our World in Data — CO2 and Greenhouse Gas Emissions
**HCD Phase:** Define — *"Name the problem precisely enough to act on it."*
**Today you will:** Assess → Clean → Visualize → Tell the Story → Write your team's Problem Statement

---

### Vocabulary you need before you start

| Term | Plain-English meaning |
|---|---|
| **Missing value** | A cell in the spreadsheet that is blank — the measurement was never collected, or doesn't apply. |
| **Outlier** | A data point far outside the normal range. Sometimes it's an error. Sometimes it's the most important data point in the whole dataset. |
| **Regional aggregate** | A row that isn't a real country — it's a *rollup* like "World," "Asia," or "European Union (27)." These will break a country-level analysis if you don't catch them. |
| **Data-ink ratio** | A design idea: every pixel of ink on a chart should carry information. Delete anything that doesn't. |
| **Finding-based title** | A chart title that states the *result*, not the *topic*. Not "CO2 by Country" — instead "China's per-person emissions have overtaken the EU's since 2005." |

> **Today's rule:** every time you drop a row, fill a blank, or pick a chart type, you must be able to say *why* in one sentence. "I felt like it" is not a justification. A data scientist's decisions are auditable.


### Warm-up discussion (no coding — 3 min)

Yesterday (Tuesday) we looked at charts that lied through design choices: truncated axes, misleading colors, cherry-picked time windows.

**Discuss with your team before you touch any code:**
1. Your dataset tracks CO2 emissions for every country since the 1700s-1800s. What's one way someone could cherry-pick this data to make a *false* claim about climate change (in either direction — overstating OR understating the problem)?
2. Who collected this data, and who might be undercounted or missing? (Hint: think about which countries have had reliable government statistics agencies for 150+ years, and which haven't.)

Write your answers in your Data Journal now — you'll need this thinking for your Data Empathy Map.


## Step 0 — Setup

Run the cell below to import your tools and load the dataset.

**Concept check:** `pandas` is the library that lets Python read and manipulate spreadsheet-like data (rows and columns). `matplotlib` and `seaborn` draw charts. We import them once, at the top, so every cell below can use them.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Make charts a readable size by default
plt.rcParams["figure.figsize"] = (9, 5)
sns.set_style("whitegrid")

# ---- LOAD THE DATASET ----
# Fallback below points to the official Our World in Data GitHub repo.
DATA_URL = "https://raw.githubusercontent.com/owid/co2-data/master/owid-co2-data.csv"

df = pd.read_csv(DATA_URL)
print("Loaded!  Shape:", df.shape)
df.head()

## Step 1 — Assess: Write a Data Audit Report

Before you clean anything, you have to know what's broken. Run each cell below and **record what you find** in the Data Audit Report template at the bottom of this section.

**Function reminders:**
- `df.info()` → data types and which columns have missing values
- `df.describe()` → mean, min, max, etc. for every numeric column (a fast way to spot impossible values, like a negative population)
- `df.isnull().sum()` → exact count of missing values per column


In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
missing_counts = df.isnull().sum().sort_values(ascending=False)
missing_pct = (missing_counts / len(df) * 100).round(1)
pd.DataFrame({"missing_count": missing_counts, "missing_pct": missing_pct}).head(20)

### Concept: "Regional aggregates" — the hidden trap in this dataset

This dataset does **not** contain only real countries. It also contains rollup rows like `"World"`, `"Asia"`, `"European Union (27)"`, and `"High-income countries"`. These are useful for some questions, but they will **wreck** a country-level comparison — imagine averaging "World" in with 190 actual countries.

**How do we tell them apart?** Real countries have a 3-letter `iso_code` (like `USA`, `IDN`, `DNK`). Regional aggregates almost always have a **blank/NaN** `iso_code`.

Run this cell to see the difference:


In [ ]:
# Rows with NO iso_code are regional aggregates, not real countries
aggregates = df[df["iso_code"].isnull()]
print("Number of aggregate rows:", len(aggregates))
print("\nExamples of aggregate 'country' names:")
print(aggregates["country"].unique()[:15])

> **Data Journal prompt (answer now):** If you left these aggregate rows in your analysis by accident, what specific number would be wrong, and who would be misled by it? Be specific — name a chart or statistic.


### Data Audit Report — fill in the blanks

> **Dataset shape:** _____ rows × _____ columns
>
> **Top 3 columns with the most missing data, and my guess for *why* each is missing:**
> 1. Column: __________ — % missing: _____ — Why: __________
> 2. Column: __________ — % missing: _____ — Why: __________
> 3. Column: __________ — % missing: _____ — Why: __________
>
> **Number of regional-aggregate rows found:** _____
>
> **One more data quality issue I noticed (duplicate rows? weird data types? impossible values?):** __________


## Step 2 — Clean: Build a documented, justified pipeline

**The ethics of cleaning, reminder from Monday:** every cleaning choice is an analytical choice. If you drop every row missing `co2`, are you dropping small poor countries at a higher rate than large rich ones? That would bias your final findings toward the countries that have always had reliable data. **Say so if it's true.**

We will build the pipeline in five small, checkable steps. Each step has a **baseline version that runs out of the box** — your job is to read it, understand it, then **tweak the choices** (which columns, which threshold, which fill strategy) to fit what your team actually needs for your Capstone question.

### Step 2a — Narrow to the columns your team actually needs

Working with all 79 columns is overwhelming and mostly irrelevant to your investigation. Baseline below keeps a working set of climate-relevant columns.

**Hint ladder (use only what you need):**
- *Nudge:* Look at the column list you saw in `df.info()`. Which ones relate to your team's research question?
- *Syntax:* `df[["col1", "col2", ...]]` selects a subset of columns.
- *Bridge:* If you want to *add* a column to the baseline list, just add its exact name (copy-paste from `df.info()` to avoid typos) into the list below.


In [ ]:
# BASELINE column set — tweak this list for your team's specific question
core_cols = [
    "country", "year", "iso_code", "population", "gdp",
    "co2", "co2_per_capita", "co2_growth_prct",
    "coal_co2", "oil_co2", "gas_co2", "cement_co2",
    "primary_energy_consumption", "energy_per_capita",
    "methane", "nitrous_oxide", "total_ghg",
    "temperature_change_from_co2", "temperature_change_from_ghg",
]

df_climate = df[core_cols].copy()
print("New shape:", df_climate.shape)
df_climate.head()

### Step 2b — Remove regional aggregates

**Justification (write this in a markdown cell of your own before moving on):** *"We are removing rows with no `iso_code` because our research question is about comparing real countries, and mixing in continent/world rollups would double-count emissions and distort our averages."*


In [ ]:
before = len(df_climate)
df_climate = df_climate[df_climate["iso_code"].notnull()].copy()
after = len(df_climate)
print(f"Removed {before - after} regional-aggregate rows. Remaining: {after}")

### Step 2c — Handle missing values

You have choices here, and they are NOT equivalent:
- `.dropna(subset=[...])` — deletes any row missing a value in the listed columns. Use when the column is essential and can't be reasonably estimated.
- `.fillna(0)` — replaces missing with zero. **Dangerous** — only correct if "missing" truly means "zero," e.g. a country that plausibly produced no cement.
- `.fillna(df[col].median())` — replaces missing with the typical value. Reduces bias from outliers, but hides the fact that data was missing at all.

**Baseline strategy below.** Your job: change ONE of these choices for a column central to your research question, and write a one-sentence justification for the change.


In [ ]:
# BASELINE strategy — tweak per the instructions above
# 1) co2 and co2_per_capita are central to almost every climate question -> drop rows missing them
df_climate = df_climate.dropna(subset=["co2", "co2_per_capita"]).copy()

# 2) gdp and population have lots of legitimate historical gaps -> leave as NaN for now (don't fake a number)
#    (We will exclude rows with missing gdp/population LATER, only in the specific analysis that needs them.)

# 3) Emissions-source breakdowns (coal/oil/gas/cement) -> fill missing with 0
#    Justification: for many small countries, "no data reported" for e.g. cement_co2 usually means
#    that source is negligible/near-zero for that country, not that the value is unknowable.
source_cols = ["coal_co2", "oil_co2", "gas_co2", "cement_co2"]
df_climate[source_cols] = df_climate[source_cols].fillna(0)

print("Shape after handling missing values:", df_climate.shape)
df_climate.isnull().sum()

### Step 2d — Check for duplicates

**Function hint:** `df.duplicated(subset=["country", "year"])` flags rows that repeat the same country-year combination — which shouldn't happen in this dataset (one row per country per year).


In [ ]:
dupes = df_climate.duplicated(subset=["country", "year"]).sum()
print("Duplicate country-year rows found:", dupes)
# If dupes > 0, uncomment the line below to drop them, keeping the first occurrence:
# df_climate = df_climate.drop_duplicates(subset=["country", "year"], keep="first")

### Step 2e — Verify your cleaning worked

**Do not skip this.** An `assert` statement stops your notebook with an error if something you expect to be true is actually false — this is how professional data scientists catch mistakes before they become published errors.


In [ ]:
assert df_climate["co2"].isnull().sum() == 0, "co2 still has missing values!"
assert df_climate["co2_per_capita"].isnull().sum() == 0, "co2_per_capita still has missing values!"
assert df_climate["iso_code"].isnull().sum() == 0, "Regional aggregates are still in the data!"
print("All checks passed. Cleaned shape:", df_climate.shape)
df_climate.describe()

### Data Limitations paragraph — fill in

> **What our cleaning process could NOT fix:** __________
>
> *(Think about: countries with no data at all for certain decades, whether small/poor nations are underrepresented after our cleaning, and what our fillna(0) choice for emissions sources might hide.)*


In [ ]:
# Save your cleaned dataset with a clear filename convention, per the syllabus.
from datetime import date
today_str = date.today().strftime("%Y%m%d")
out_name = f"climate_cleaned_{today_str}.csv"
df_climate.to_csv(out_name, index=False)
print("Saved:", out_name, "— now upload this to your GitHub repo.")

## Step 3 — Visualize: Four publication-quality charts

**Rule from yesterday:** every chart needs (1) a **finding-based title** — stating the result, not the topic — (2) labeled axes with units, and (3) a 2-sentence written interpretation. A chart with no interpretation is just decoration.

You will build one of each: **histogram, bar chart, line chart, scatter plot.** For each, read the concept, run the baseline, then tweak it.


### Exploratory Data Analysis: Variables Explored

For our Exploratory Data Analysis, we are examining the following variables:

*   `co2_per_capita`: CO2 emissions per person.
*   `co2`: Total CO2 emissions.
*   `year`: The year of observation, used for temporal trends.
*   `gdp_per_capita`: Gross Domestic Product per person, indicating economic output.
*   `population`: The total population of a country.
*   `primary_energy_consumption`: Total primary energy consumed.
*   `methane`, `nitrous_oxide`: Other greenhouse gases.

These variables are analyzed through summary statistics (`df.describe()`, `df.info()`, `df.isnull().sum()`) and visualized in the charts below.

### Chart 1 — Histogram: What does the distribution of per-person emissions look like?

**Concept:** A histogram groups a continuous number (here, `co2_per_capita`) into bins and counts how many rows fall in each bin. It answers: *is this measurement typical for most countries, or do a few extreme countries dominate?*

**Function hint:** `sns.histplot(data=df, x="column_name", bins=30)`

**What to look for:** Is the shape symmetric, or does it have a long tail (skewed)? A long right tail means a few countries emit *far* more per person than everyone else.


In [ ]:
# Use the MOST RECENT year available so the comparison is fair (not mixing 1850 with 2020)
latest_year = df_climate["year"].max()
df_latest = df_climate[df_climate["year"] == latest_year]

# BASELINE
plt.figure()
sns.histplot(data=df_latest, x="co2_per_capita", bins=30)
plt.xlabel("CO2 emissions per person (tonnes)")
plt.ylabel("Number of countries")
plt.title(f"FILL IN: state your finding here, e.g. 'Most countries emit under X tonnes per person in {latest_year}'")
plt.show()

# HINT LADDER if you want to improve this chart:
#  Nudge: is bins=30 too coarse or too fine to see the shape clearly? Try a few values.
#  Syntax: change the number inside bins=... and re-run.
#  Bridge: try df_latest["co2_per_capita"] < 30  to filter out extreme outliers and re-plot,
#          so the "typical" countries are easier to see. Decide: should you keep or remove
#          those outlier countries? Justify your choice in the interpretation below.

> **Interpretation (2 sentences, with a real statistic and units):** __________


### Chart 2 — Bar chart: Which countries/regions stand out for total emissions?

**Concept:** A bar chart compares a number across categories (here, countries). Use it for comparison, not for a continuous trend over time.

**Function hint:** `sns.barplot(data=..., x="co2", y="country", order=...)` — use `order=` to sort bars biggest-to-smallest so the finding is instantly visible. Use **horizontal** bars (`y="country"`) because country names are long.


In [ ]:
# BASELINE — top 10 emitting countries in the latest year, by TOTAL co2 (not per-capita)
top10 = df_latest.nlargest(10, "co2")

plt.figure()
sns.barplot(data=top10, x="co2", y="country",
            order=top10.sort_values("co2", ascending=False)["country"])
plt.xlabel("Total CO2 emissions (million tonnes)")
plt.ylabel("")
plt.title(f"FILL IN your finding, e.g. 'Just N countries account for over half of {latest_year} emissions'")
plt.show()

# HINT LADDER:
#  Nudge: total emissions favor big/populous countries. Is that the fairest comparison for YOUR question?
#  Syntax: swap x="co2" for x="co2_per_capita" and re-run nlargest on that column instead.
#  Bridge: does the ranking change a lot between total vs per-capita? That difference IS a finding —
#          write about it.

> **Interpretation (2 sentences, with a real statistic and units):** __________


### Chart 3 — Line chart: How has something changed over time?

**Concept:** A line chart is for a **truly continuous time series** — use it only when the x-axis is time.

**Function hint:** `sns.lineplot(data=..., x="year", y="co2_per_capita", hue="country")`


In [ ]:
# BASELINE — pick a few countries to compare over the full time range
countries_to_compare = ["United States", "China", "India", "Nigeria"]  # <-- TWEAK this list for your team
df_trend = df_climate[df_climate["country"].isin(countries_to_compare)]

plt.figure()
sns.lineplot(data=df_trend, x="year", y="co2_per_capita", hue="country")
plt.xlabel("Year")
plt.ylabel("CO2 emissions per person (tonnes)")
plt.title("FILL IN your finding, e.g. 'US per-capita emissions have fallen since 2005 while China's rose'")
plt.legend(title="Country")
plt.show()

# HINT LADDER:
#  Nudge: are you comparing countries with very different population sizes? Per-capita already
#         adjusts for that -- is that the right choice for this specific comparison?
#  Syntax: to zoom into a recent period only, filter first: df_trend[df_trend["year"] >= 1990]
#  Bridge: which country's LINE SHAPE tells the most surprising story? Zoom into just that one
#          country's trend using a second small plot.

> **Interpretation (2 sentences, with a real statistic and units):** __________


### Chart 4 — Scatter plot: Does wealth relate to emissions?

**Concept:** A scatter plot shows the relationship between two continuous variables — each dot is one country. This is the chart type we'll build on heavily in Week 3 (correlation/regression).

**Requirement:** at least one of your four charts must **challenge an assumption a reader might bring to the topic.** This is a strong candidate — many people assume "richer = always more emissions," but per-capita relationships are more nuanced.

**Function hint:** `sns.scatterplot(data=..., x="gdp_per_capita", y="co2_per_capita")`


In [ ]:
# gdp_per_capita isn't a column yet -- we have to CREATE it. This is a normal, expected step.
df_latest = df_latest.copy()
df_latest["gdp_per_capita"] = df_latest["gdp"] / df_latest["population"]

# Drop rows where we can't compute this (missing gdp or population) -- ONLY for this specific chart
df_scatter = df_latest.dropna(subset=["gdp_per_capita", "co2_per_capita"])

plt.figure()
sns.scatterplot(data=df_scatter, x="gdp_per_capita", y="co2_per_capita", alpha=0.6)
plt.xlabel("GDP per person (US dollars)")
plt.ylabel("CO2 emissions per person (tonnes)")
plt.title("FILL IN your finding here")
plt.show()

# HINT LADDER:
#  Nudge: is the cloud of dots too squeezed on the left? Money often needs a LOG scale to see clearly.
#  Syntax: plt.xscale("log")   <- add this line before plt.show() and re-run
#  Bridge: which countries are the clearest EXCEPTIONS to the overall pattern (high GDP, low emissions
#          per person, or the reverse)? Label 2-3 of them using plt.annotate() and discuss why they
#          might be exceptions in your interpretation.

> **Interpretation (2 sentences, with a real statistic and units) — and explain what assumption this chart challenges:** __________


### Chart 5 — Correlation Heatmap: How do variables relate to each other?

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(
    df_latest[[
        "co2_per_capita",
        "gdp_per_capita",
        "population",
        "primary_energy_consumption",
        "methane",
        "nitrous_oxide",
    ]].corr(),
    annot=True,
    cmap="coolwarm",
    fmt=".2f"
)
plt.title("Correlation Matrix of Key Variables in Latest Year")
plt.show()


NameError: name 'df_latest' is not defined

<Figure size 1000x800 with 0 Axes>

> **Interpretation (2 sentences, with a real statistic and units):** __________

## Step 4 — Story: Write a 150-word Data Brief

**Audience:** a city council member who has never seen your raw data and does not know what "per capita" means unless you explain it in context.

**Requirements:**
- 150 words, no more
- Your 3 most important findings, in plain language
- At least 2 specific statistics with units
- No jargon left unexplained (if you must use a term like "per capita," define it in the same sentence)
- End with what this means for a real decision

Draft it in the cell below (as a markdown cell, not code) or directly in your team doc.


### Your Data Brief (150 words):

> _Write it here._


## HCD Design Sprint #2 — Define: Write Your Team's Data Problem Statement

This is a **team** deliverable, reviewed and approved by your instructor before Thursday. Fill in every blank — vague blanks will be sent back for revision.

> **Our investigation asks:** _[specific, answerable question that can be addressed with our dataset]_
>
> **The dataset we are using is:** _[name, source, year collected, unit of analysis]_
> *(e.g., "Our World in Data CO2 and Greenhouse Gas Emissions dataset, updated 2024, one row per country per year")*
>
> **The key variable we are analyzing is:** _[variable name, what it measures, its range and units]_
>
> **Our investigation matters because:** _[human consequence]_ — without better understanding of this pattern, _[who]_ will continue to face _[harm or inequity]_.
>
> **We define a meaningful finding as:** _[specific threshold or pattern that would warrant a policy recommendation]_
> *(e.g., "a country whose per-capita emissions grew more than 10% while its climate-vulnerability indicators worsened")*

---

**Before you move on, check:** Does your question actually match a column that exists in `df_climate`? If not, revise the question now — don't discover this Thursday.


## Data Journal — Exit Ticket (5 min, do this now)

Answer in your Data Journal, not here:

1. What dataset or technique did I work with today?
2. What pattern or anomaly surprised me?
3. Who is represented in this data — and who is invisible?
4. What would an ethical data scientist do with this information?


## Submission checklist

- [ ] Data Audit Report filled in
- [ ] Cleaning pipeline runs top-to-bottom with no errors, all `assert` checks pass
- [ ] `climate_cleaned_YYYYMMDD.csv` saved and pushed to your team's GitHub repo
- [ ] 4 charts, each with a finding-based title, labeled axes, and a written interpretation
- [ ] 150-word Data Brief
- [ ] Data Problem Statement — every blank filled, submitted for instructor review
- [ ] Data Journal exit ticket complete

**Notebook submitted to course LMS at the end of session, per syllabus.**


---
# PART B — Week 4: Machine Learning, Classification & Model Comparison + HCD Design Sprint #4
---

# Week 4 — Climate Change Team
## Machine Learning: Classification & Model Comparison (End-to-End)

**Dataset:** Our World in Data — CO2 and Greenhouse Gas Emissions
**HCD Phase:** Prototype — *"Build a testable solution and find out where it fails."*

**Important note before you start:** CO2 emissions data doesn't come with a ready-made yes/no label. In real data science, you often have to **build** the thing you're trying to predict. Today, your instructor has defined it for you:

> **Our prediction target: "High-Emitter Year."** A country-year is labeled **1 (High-Emitter)** if that country's `co2_per_capita` in that year is **above the median** for all country-years in the dataset, and **0 (Not High-Emitter)** if it's at or below the median.

This is a **derived label** — we built it ourselves from a continuous number by picking a cutoff (the median). That is itself an analytical decision, and like every analytical decision, it has consequences you need to be able to explain.

---

### Vocabulary you need before you start

| Term | Plain-English meaning |
|---|---|
| **Feature** | An input the model uses to make its guess (e.g., a country's GDP per person). |
| **Label / target** | The thing you're trying to predict (here: High-Emitter or not). |
| **Train/test split** | You teach the model using one chunk of the data, then quiz it on a *different* chunk it has never seen — this is the only honest way to know if it actually learned something. |
| **Baseline model** | The "dumbest possible guess" — e.g., always guess the most common label. If your fancy model can't beat this, it isn't learning anything useful. |
| **Confusion matrix** | A table of what the model got right and wrong, broken into 4 outcomes (explained below). |
| **Fairness check** | Testing whether a model is equally accurate across different groups — or whether it fails one group more than another. |


### Warm-up discussion (no coding — 3 min)

Recall the COMPAS algorithm from Monday: a recidivism-prediction model that was "accurate on average" but nearly twice as likely to wrongly flag Black defendants as high-risk.

**Discuss with your team:**
1. If we build a model that flags countries as "High-Emitter," who could be harmed by a **false positive** (wrongly flagged) vs. a **false negative** (wrongly NOT flagged)? Think about what a "flag" like this might be used for in the real world — climate financing, trade penalties, media attention.
2. Our derived label uses a **global median** across all countries and all years since the 1800s. What historical unfairness might that bake into the label itself, before the model even trains?

Record your answers in your Data Journal.


## Step 0 — Setup: Load your Week 2 cleaned dataset

**Reminder:** this builds directly on the cleaning work your team did in Week 2. If you don't have your saved CSV handy, the fallback line reloads and re-cleans from the original source.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, recall_score

plt.rcParams["figure.figsize"] = (7, 5)
sns.set_style("whitegrid")

# ---- TRY to load your Week 2 cleaned file first; fall back to a fresh clean if not found ----
import os
CLEANED_FILE = "climate_cleaned_YYYYMMDD.csv"   # <-- replace with YOUR team's actual saved filename

if os.path.exists(CLEANED_FILE):
    df = pd.read_csv(CLEANED_FILE)
    print("Loaded your Week 2 cleaned file:", df.shape)
else:
    print("Cleaned file not found -- reloading and re-cleaning from source (Week 2 steps, condensed).")
    RAW_URL = "https://raw.githubusercontent.com/owid/co2-data/master/owid-co2-data.csv"
    raw = pd.read_csv(RAW_URL)
    core_cols = ["country", "year", "iso_code", "population", "gdp", "co2", "co2_per_capita",
                 "co2_growth_prct", "coal_co2", "oil_co2", "gas_co2", "cement_co2",
                 "primary_energy_consumption", "energy_per_capita", "methane", "nitrous_oxide",
                 "total_ghg", "temperature_change_from_co2", "temperature_change_from_ghg"]
    df = raw[core_cols].copy()
    df = df[df["iso_code"].notnull()].copy()
    df = df.dropna(subset=["co2", "co2_per_capita"]).copy()
    df[["coal_co2","oil_co2","gas_co2","cement_co2"]] = df[["coal_co2","oil_co2","gas_co2","cement_co2"]].fillna(0)
    print("Re-cleaned shape:", df.shape)

df.head()

Cleaned file not found -- reloading and re-cleaning from source (Week 2 steps, condensed).
Re-cleaned shape: (22945, 19)


,country,year,iso_code,population,gdp,co2,co2_per_capita,co2_growth_prct,coal_co2,oil_co2,gas_co2,cement_co2,primary_energy_consumption,energy_per_capita,methane,nitrous_oxide,total_ghg,temperature_change_from_co2,temperature_change_from_ghg
199,Afghanistan,1949,AFG,7356890.0,NaN,0.015,0.002,NaN,0.015,0.000,0.0,0.0,NaN,NaN,7.729,2.162,18.667,0.0,0.001
200,Afghanistan,1950,AFG,7776180.0,9.421400e+09,0.084,0.011,475.000,0.021,0.063,0.0,0.0,NaN,NaN,7.879,2.231,19.869,0.0,0.001
201,Afghanistan,1951,AFG,7879343.0,9.692280e+09,0.092,0.012,8.696,0.026,0.066,0.0,0.0,NaN,NaN,7.973,2.292,21.069,0.0,0.001
202,Afghanistan,1952,AFG,7987784.0,1.001732e+10,0.092,0.011,0.000,0.032,0.060,0.0,0.0,NaN,NaN,8.073,2.365,22.094,0.0,0.001
203,Afghanistan,1953,AFG,8096703.0,1.063052e+10,0.106,0.013,16.000,0.038,0.068,0.0,0.0,NaN,NaN,8.186,2.447,23.256,0.0,0.001


## Step 1 — Define the Prediction Problem

We build the "High-Emitter Year" label from `co2_per_capita` using the **median** as the cutoff.

**Concept check:** why the *median* and not the *mean*? The mean is dragged upward by a handful of extremely high-emitting countries (outliers). The median is the true "middle" value — half of all country-years are above it, half below. That makes for a more balanced label.


In [ ]:
median_value = df["co2_per_capita"].median()
print(f"Median CO2 per capita across the dataset: {median_value:.2f} tonnes")

df["high_emitter"] = (df["co2_per_capita"] > median_value).astype(int)

print("\nClass balance (this should be close to 50/50, since we used the median):")
print(df["high_emitter"].value_counts(normalize=True).round(3))

> **Data Journal prompt:** A country that was a "High-Emitter" in 1950 might not be one today, and vice versa. Pick one real country from the data and check: has its `high_emitter` label changed over time? What does that tell you about treating this as a fixed category versus something that changes?


## Step 2 — Feature Engineering

**Critical concept — data leakage:** We CANNOT use `coal_co2`, `oil_co2`, `gas_co2`, or `total_ghg` as features. Why? Because those numbers are literally the ingredients that `co2_per_capita` is built from — using them would be like "predicting" whether someone is tall using their height in a different unit. The model would look impressively accurate and teach you nothing.

**Instead, we'll use features that are *related* to emissions but not derived directly from them:**
- `gdp_per_capita` (created below) — a country's wealth per person
- `population` — country size
- `primary_energy_consumption` — total energy use (related to, but not identical to, emissions)
- `energy_per_capita` — energy use per person
- `methane`, `nitrous_oxide` — different greenhouse gases (not CO2 itself)
- `year` — captures broad historical trends


In [ ]:
# Create gdp_per_capita
df["gdp_per_capita"] = df["gdp"] / df["population"]

feature_cols = ["gdp_per_capita", "population", "primary_energy_consumption",
                "energy_per_capita", "methane", "nitrous_oxide", "year"]

# HINT LADDER if you want to add/remove features:
#  Nudge: does adding a feature make conceptual sense, or could it be leakage (derived from co2)?
#  Syntax: just add/remove the column name (string) from feature_cols above.
#  Bridge: after changing features, re-run every cell below this one in order.

# Drop rows missing any feature we need
model_df = df.dropna(subset=feature_cols + ["high_emitter"]).copy()
print("Rows available for modeling:", len(model_df))
print("Rows dropped due to missing features:", len(df) - len(model_df))

model_df[feature_cols].describe()

> **Written justification (fill in):** We dropped _____ rows because they were missing at least one of our chosen features. This likely affects _____ (which kinds of countries/years?) more than others, which means our model's results should be read with that limitation in mind.


## Step 3 — Train/Test Split

**Concept check:** we train the model on 80% of the data and test it on the other 20% — data the model has **never seen**. If we tested it on data it already memorized, we'd have no idea if it actually learned a real pattern or just memorized answers.

**Function hint:** `train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)` — `stratify=y` makes sure both the train and test sets have a similar High-Emitter/Not split, so the test is fair.


In [ ]:
X = model_df[feature_cols]
y = model_df["high_emitter"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training rows:", len(X_train), "| Test rows:", len(X_test))
print("\nClass balance in training set:")
print(y_train.value_counts(normalize=True).round(3))
print("\nClass balance in test set:")
print(y_test.value_counts(normalize=True).round(3))

### Standardize the features

**Concept check:** some models (like k-NN) measure "distance" between data points. If `population` is in the hundreds of millions and `gdp_per_capita` is in the thousands, population will completely dominate the distance calculation just because its numbers are bigger — not because it's actually more important. Standardizing rescales every feature to the same range so each one gets a fair vote.


In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Note: we fit the scaler ONLY on training data, then apply it to test data.
# This prevents the test set from "leaking" information into training.

## Step 4 — Establish a Baseline (the "dumbest possible guess")

Before we train any real model, we need a floor to beat. `DummyClassifier(strategy="most_frequent")` always predicts whatever label is most common — it doesn't look at the features at all. If our real models can't beat this, they aren't learning anything.


In [ ]:
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train_scaled, y_train)
baseline_pred = baseline.predict(X_test_scaled)
baseline_acc = accuracy_score(y_test, baseline_pred)

print(f"Baseline accuracy (always guess the most common class): {baseline_acc:.3f}")
print("Any real model MUST beat this number, or it has learned nothing useful.")

## Step 5 — Train and Compare Multiple Algorithms

We'll train **four** classifiers on the exact same data and compare them fairly. Each has a different way of "thinking":

- **Logistic Regression** — finds a smooth mathematical boundary between the two classes based on weighted combinations of features. Fast, and easy to explain ("more energy_per_capita increases the odds of High-Emitter").
- **k-Nearest Neighbors (k-NN)** — classifies a country-year by looking at the *k* most similar rows (its "neighbors") and taking a majority vote.
- **Decision Tree** — asks a sequence of yes/no questions ("is `energy_per_capita` above X? then is `gdp_per_capita` above Y?") until it reaches an answer. Easy to visualize as a flowchart.
- **Random Forest** — trains *many* decision trees on random subsets of the data and lets them vote together. Usually more accurate and less prone to memorizing quirks than a single tree.

**Baseline hyperparameters are given below. Your job: tweak at least ONE hyperparameter per model and observe what changes.**


In [ ]:
# BASELINE MODELS -- read the comments, then tweak the marked hyperparameters

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

models = {
    "Logistic Regression": LogisticRegression(random_state=42, max_iter=1000),

    # HINT: n_neighbors (k) controls how many "neighbors" vote.
    #   Small k (e.g. 3) -> very sensitive to noise / individual points ("overfit")
    #   Large k (e.g. 50) -> smoother, but may blur real distinctions ("underfit")
    #   TWEAK: try k=3, k=15, k=50 and compare accuracy below.
    "k-Nearest Neighbors": KNeighborsClassifier(n_neighbors=15),

    # HINT: max_depth controls how many yes/no questions the tree can ask.
    #   Shallow tree (depth=2) -> simple, explainable, may miss real patterns
    #   Deep tree (depth=None) -> can memorize the training data ("overfit") and fail on new data
    #   TWEAK: try max_depth=2, max_depth=6, max_depth=None and compare.
    "Decision Tree": DecisionTreeClassifier(max_depth=4, random_state=42),

    # HINT: n_estimators is how many trees vote together. More trees = more stable,
    #   but slower. TWEAK: try n_estimators=10 vs n_estimators=200.
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42),

    # Added Gradient Boosting Classifier
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
}

results = {}
predictions = {}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    pred = model.predict(X_test_scaled)
    predictions[name] = pred
    results[name] = {
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred),
        "Recall": recall_score(y_test, pred),
        "F1 Score": f1_score(y_test, pred)
    }

    print(
        f"{name:25s} | "
        f"Accuracy = {results[name]['Accuracy']:.3f} | "
        f"Precision = {results[name]['Precision']:.3f} | "
        f"Recall = {results[name]['Recall']:.3f} | "
        f"F1 = {results[name]['F1 Score']:.3f}"
    )

NameError: name 'X_train_scaled' is not defined

In [ ]:
# comparing everything in a new table
results_df = (
    pd.DataFrame(results)
    .T
    .sort_values(by="F1 Score", ascending=False)
)

results_df.style \
    .background_gradient(
        cmap="Greens",
        subset=["Accuracy", "Precision", "Recall", "F1 Score"]
    ) \
    .format("{:.3f}")

## Step 6 — Evaluate & Compare in Detail

**Concept check — the Confusion Matrix:**

| | Predicted: Not High-Emitter | Predicted: High-Emitter |
|---|---|---|
| **Actually: Not High-Emitter** | True Negative (correct) | False Positive (wrongly flagged) |
| **Actually: High-Emitter** | False Negative (missed it) | True Positive (correct) |

- **Precision** = of everything the model flagged as High-Emitter, how much was actually correct?
- **Recall** = of everything that WAS actually High-Emitter, how much did the model catch?

Which one matters more depends on the real-world use of the flag — think back to your warm-up discussion.


In [ ]:
for name, pred in predictions.items():
    print(f"===== {name} =====")
    print(classification_report(y_test, pred, target_names=["Not High-Emitter", "High-Emitter"]))
    print()

> **Written comparison (fill in):** The model with the highest accuracy is __________. The model with the highest recall for High-Emitter is __________. These are the same / different model *(circle one)* — if different, explain in one sentence why a model can be "more accurate overall" but worse at catching the specific class you care about.


## Step 7 — Fairness Check

This dataset doesn't include a demographic column like race or gender. But it DOES let us check something equally important for a global climate model: **does the model perform equally well for rich countries versus poor countries?** If a "High-Emitter" flag is going to influence real decisions (climate financing, trade policy, media narratives), a model that's much less accurate for low-income countries would be a serious problem — it could unfairly blame or unfairly excuse specific groups of nations based on model error, not reality.

**We'll split countries into income tiers using `gdp_per_capita`**, a proxy already available in our features.

**Note this is a proxy, not a perfect measure** — a country's income tier changes over time, and per-capita GDP is an imperfect stand-in for a country's actual development context. Say so in your writeup.


In [ ]:
# Recover the test set's gdp_per_capita (dropped when we scaled to a plain array)
test_gdp_pc = X_test["gdp_per_capita"].reset_index(drop=True)

# Split into income tiers using quartiles (roughly equal-sized groups)
income_tier = pd.qcut(test_gdp_pc, q=3, labels=["Low income tier", "Middle income tier", "High income tier"])

y_test_reset = y_test.reset_index(drop=True)

# Use your best-performing model from Step 6 -- CHANGE this variable name to match your top model
best_model_name = "Random Forest"   # <-- TWEAK: set this to whichever model had the best results
best_pred = pd.Series(predictions[best_model_name]).reset_index(drop=True)

fairness_df = pd.DataFrame({
    "income_tier": income_tier,
    "actual": y_test_reset,
    "predicted": best_pred,
})

print(f"Fairness check for: {best_model_name}\n")
for tier in ["Low income tier", "Middle income tier", "High income tier"]:
    subset = fairness_df[fairness_df["income_tier"] == tier]
    acc = accuracy_score(subset["actual"], subset["predicted"])
    rec = recall_score(subset["actual"], subset["predicted"], pos_label=1, zero_division=0)
    print(f"{tier:20s} | n={len(subset):4d} | accuracy = {acc:.3f} | recall(High-Emitter) = {rec:.3f}")

> **Fairness write-up (fill in):**
>
> Is performance roughly equal across income tiers, or does the model fail one group more than another? __________
>
> If there IS a gap, what real-world consequence could that have if this model informed an actual policy or financing decision? __________
>
> What is ONE thing you could try to reduce that gap? *(e.g., more balanced training data, a different feature set, a different threshold per group)* __________


## Step 8 — Recommendation

Write 3 sentences, following this pattern from the syllabus:

> "I recommend **[model]** for this application because **[metric]** is most important, given that a false negative here means **[human consequence]** and a false positive means **[human consequence]**."

Fill in your own answer below:

> **Our recommendation:** __________


## HCD Design Sprint #4 — Prototype: Assemble Your Capstone Prototype

Your team's Capstone prototype must now include, at minimum:

1. A cleaned dataset with a documented cleaning log *(from Week 2)*
2. 3 final visualizations supporting your central finding *(from Week 2)*
3. A regression or classification analysis with fully interpreted results *(Week 3 regression + this notebook's classification)*
4. A draft policy recommendation, in this exact format:

> "Based on our data analysis, we recommend that **[institution]** take the following action, because our evidence shows **[specific finding]** implies **[specific consequence]** for **[population]** if unaddressed."

Fill in your team's draft below:

> **Our policy recommendation:** __________

**Peer review (trade with another team):**
- The strongest evidence in this prototype is... __________
- The biggest gap between the data and the recommendation is... __________
- One more analysis that would strengthen this is... __________


## Data Journal — Exit Ticket

1. What does it mean for an algorithm to be fair? Is fairness the same as accuracy?
2. Who should be allowed to define what counts as a fair prediction for a model like this — the model developer, the company, the government, or the countries the model affects?
3. If you were a policymaker and someone showed you this model's results, what is the FIRST question you would ask before trusting it?


## Submission checklist

- [ ] Target variable (`high_emitter`) defined with justification for the median cutoff
- [ ] Feature set chosen with leakage explicitly avoided and justified
- [ ] Train/test split performed with `stratify=y`
- [ ] Baseline (dummy) accuracy recorded as the floor to beat
- [ ] All 4 models trained; at least one hyperparameter tweaked per model
- [ ] Confusion matrices and classification reports compared across all models
- [ ] Fairness check completed across income tiers, with a written response
- [ ] Model recommendation written in the required sentence format
- [ ] HCD Prototype assembled and peer-reviewed
- [ ] Data Journal exit ticket complete
- [ ] Results CSV pushed to GitHub

**Notebook submitted to course LMS / GitHub, per syllabus.**
